<a href="https://colab.research.google.com/github/hjk9655/health-checkup-analysis/blob/26.07.06-test/self_practice_workshop_1A_python_pandas_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 바이오 빅데이터 분석 (1A): Python/Pandas Bridge + Visualization Practice

## 0. 필요한 패키지 설치 (처음 한번만 하면 됨)

In [1]:
# !ls 현재 파일 보여주는 것
!ls sample_data

anscombe.json		      mnist_test.csv
california_housing_test.csv   mnist_train_small.csv
california_housing_train.csv  README.md


In [2]:
!python --version
# !pip list 패키지가 뭐 설치되어있는지

Python 3.12.13


In [1]:
# 밑에 3가지 패키지를 설치하는 것임(mlbi-lab, scikit-network, statannotations)
!pip install mlbi-lab scikit-network statannotations

In [2]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr
from scipy.stats import ttest_ind, mannwhitneyu, f_oneway, kruskal

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn import cluster, mixture
from sklearn.neighbors import kneighbors_graph

from sknetwork.clustering import Louvain
from mlbi.datasets import load_data

In [3]:
load_data()

You can select one of:
  cancerseek
  ccle-ctrpv2
  heart_failure
  hotel_bookings
  house_price
  labor_force
  metabric
  scores
  tcga-brca
  time-series
  time-series2


## 1. Data Frame의 구조와 활용법 실습

### 분석 관점에서 보는 DataFrame

In [9]:
# 예제의 DataFrame을 만들고 연습하기

example_df = pd.DataFrame({
    'sameple_id' : ['S1', 'S2', 'S3', 'S4'],
    'subtype' : ['Lum.A', 'Lum.B', 'Basal', 'Her2'],
    'ESR1': [12.4, 8.1, 1.2, 4.5],
    'ERBB2': [3.1, 4.0, 2.8, 14.2],
    'MKI67': [2.3, 7.8, 9.1, 6.5]
})

example_df

,sameple_id,subtype,ESR1,ERBB2,MKI67
0,S1,Lum.A,12.4,3.1,2.3
1,S2,Lum.B,8.1,4.0,7.8
2,S3,Basal,1.2,2.8,9.1
3,S4,Her2,4.5,14.2,6.5


In [10]:
example_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   sameple_id  4 non-null      object 
 1   subtype     4 non-null      object 
 2   ESR1        4 non-null      float64
 3   ERBB2       4 non-null      float64
 4   MKI67       4 non-null      float64
dtypes: float64(3), object(2)
memory usage: 292.0+ bytes


In [12]:
example_df['ESR1'].mean()

np.float64(6.55)

### 빠른 복습: 한 열과 여러 열 선택

`df['ESR1']`은 한 열을 `Series`로 가져오고, `df[['ESR1', 'ERBB2']]`는 여러 열을 `DataFrame`으로 가져오기.  
대괄호가 하나인지 두 개인지에 따라 결과의 타입이 달라진다는 점을 자주 확인할 것.

In [20]:
print(type(example_df['ESR1']))
print(type(example_df[['ESR1', 'ERBB2']]))

print(type(example_df[['ESR1']]))

example_df[['ESR1', 'ERBB2']].std()

<class 'pandas.core.series.Series'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


,0
ESR1,4.811445
ERBB2,5.473801


### 빠른 복습: 조건으로 샘플 고르기

아래 예시는 `ESR1` 발현량이 높은 샘플만 고르는 코드임.  
실제 분석에서는 특정 subtype, 특정 stage, 특정 발현량 기준에 맞는 샘플을 고를 때 같은 방식을 쓴다.